In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sn
%matplotlib inline
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.model_selection import KFold, train_test_split, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import AdaBoostClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

# DATA

In [ ]:
#Upload File CSV
from google.colab import files
files.upload()

Saving alzheimer_oasis_dataset.csv to alzheimer_oasis_dataset.csv


{'alzheimer_oasis_dataset.csv': b'\xef\xbb\xbfSubject ID,MRI ID,Group,Visit,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF\r\nOAS2_0001,OAS2_0001_MR1,Nondemented,1,0,M,R,87,14,2,27,0,1987,0.696,0.883\r\nOAS2_0001,OAS2_0001_MR2,Nondemented,2,457,M,R,88,14,2,30,0,2004,0.681,0.876\r\nOAS2_0002,OAS2_0002_MR1,Demented,1,0,M,R,75,12,,23,0.5,1678,0.736,1.046\r\nOAS2_0002,OAS2_0002_MR2,Demented,2,560,M,R,76,12,,28,0.5,1738,0.713,1.010\r\nOAS2_0002,OAS2_0002_MR3,Demented,3,1895,M,R,80,12,,22,0.5,1698,0.701,1.034\r\nOAS2_0004,OAS2_0004_MR1,Nondemented,1,0,F,R,88,18,3,28,0,1215,0.710,1.444\r\nOAS2_0004,OAS2_0004_MR2,Nondemented,2,538,F,R,90,18,3,27,0,1200,0.718,1.462\r\nOAS2_0005,OAS2_0005_MR1,Nondemented,1,0,M,R,80,12,4,28,0,1689,0.712,1.039\r\nOAS2_0005,OAS2_0005_MR2,Nondemented,2,1010,M,R,83,12,4,29,0.5,1701,0.711,1.032\r\nOAS2_0005,OAS2_0005_MR3,Nondemented,3,1603,M,R,85,12,4,30,0,1699,0.705,1.033\r\nOAS2_0007,OAS2_0007_MR1,Demented,1,0,M,R,71,16,,28,0.5,1357,0.748,1.293\r\nOAS2_0007,O

In [ ]:
data = pd.read_csv('alzheimer_oasis_dataset.csv')

In [ ]:
x = data.drop('Group', axis=1)
y = data['Group']

In [ ]:
data.head()

,Subject ID,MRI ID,Group,Visit,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,OAS2_0001,OAS2_0001_MR1,Nondemented,1,0,M,R,87,14,2.0,27.0,0.0,1987,0.696,0.883
1,OAS2_0001,OAS2_0001_MR2,Nondemented,2,457,M,R,88,14,2.0,30.0,0.0,2004,0.681,0.876
2,OAS2_0002,OAS2_0002_MR1,Demented,1,0,M,R,75,12,NaN,23.0,0.5,1678,0.736,1.046
3,OAS2_0002,OAS2_0002_MR2,Demented,2,560,M,R,76,12,NaN,28.0,0.5,1738,0.713,1.010
4,OAS2_0002,OAS2_0002_MR3,Demented,3,1895,M,R,80,12,NaN,22.0,0.5,1698,0.701,1.034


In [ ]:
data.shape

(373, 15)

In [ ]:
data.columns

Index(['Subject ID', 'MRI ID', 'Group', 'Visit', 'MR Delay', 'M/F', 'Hand',
       'Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF'],
      dtype='object')

In [ ]:
data.describe

<bound method NDFrame.describe of     Subject ID         MRI ID        Group  Visit  MR Delay M/F Hand  Age  \
0    OAS2_0001  OAS2_0001_MR1  Nondemented      1         0   M    R   87   
1    OAS2_0001  OAS2_0001_MR2  Nondemented      2       457   M    R   88   
2    OAS2_0002  OAS2_0002_MR1     Demented      1         0   M    R   75   
3    OAS2_0002  OAS2_0002_MR2     Demented      2       560   M    R   76   
4    OAS2_0002  OAS2_0002_MR3     Demented      3      1895   M    R   80   
..         ...            ...          ...    ...       ...  ..  ...  ...   
368  OAS2_0185  OAS2_0185_MR2     Demented      2       842   M    R   82   
369  OAS2_0185  OAS2_0185_MR3     Demented      3      2297   M    R   86   
370  OAS2_0186  OAS2_0186_MR1  Nondemented      1         0   F    R   61   
371  OAS2_0186  OAS2_0186_MR2  Nondemented      2       763   F    R   63   
372  OAS2_0186  OAS2_0186_MR3  Nondemented      3      1608   F    R   65   

     EDUC  SES  MMSE  CDR  eTIV   nWBV   

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 373 entries, 0 to 372
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Subject ID  373 non-null    object 
 1   MRI ID      373 non-null    object 
 2   Group       373 non-null    object 
 3   Visit       373 non-null    int64  
 4   MR Delay    373 non-null    int64  
 5   M/F         373 non-null    object 
 6   Hand        373 non-null    object 
 7   Age         373 non-null    int64  
 8   EDUC        373 non-null    int64  
 9   SES         354 non-null    float64
 10  MMSE        371 non-null    float64
 11  CDR         373 non-null    float64
 12  eTIV        373 non-null    int64  
 13  nWBV        373 non-null    float64
 14  ASF         373 non-null    float64
dtypes: float64(5), int64(5), object(5)
memory usage: 43.8+ KB


In [ ]:
from sklearn.preprocessing import LabelEncoder

# DATA PREPROCESSING

In [ ]:
lab=LabelEncoder()

In [ ]:
data = data.drop(data[['Subject ID','MRI ID', 'Visit']], axis=1)
data.head()

,Group,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,Nondemented,0,M,R,87,14,2.0,27.0,0.0,1987,0.696,0.883
1,Nondemented,457,M,R,88,14,2.0,30.0,0.0,2004,0.681,0.876
2,Demented,0,M,R,75,12,NaN,23.0,0.5,1678,0.736,1.046
3,Demented,560,M,R,76,12,NaN,28.0,0.5,1738,0.713,1.010
4,Demented,1895,M,R,80,12,NaN,22.0,0.5,1698,0.701,1.034


In [ ]:
data.Group.value_counts()

Nondemented    190
Demented       146
Converted       37
Name: Group, dtype: int64

In [ ]:
#menggabungkan data kelas converted ke demented
data['Group']=data['Group'].replace(['Converted'],['Demented'])

In [ ]:
data.Group.value_counts()

Nondemented    190
Demented       183
Name: Group, dtype: int64

In [ ]:
data['M/F'] = lab.fit_transform(data['M/F'])
data['Group'] = lab.fit_transform (data['Group'])
data['Hand'] = lab.fit_transform(data['Hand'])
data.head()

,Group,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,1,0,1,0,87,14,2.0,27.0,0.0,1987,0.696,0.883
1,1,457,1,0,88,14,2.0,30.0,0.0,2004,0.681,0.876
2,0,0,1,0,75,12,NaN,23.0,0.5,1678,0.736,1.046
3,0,560,1,0,76,12,NaN,28.0,0.5,1738,0.713,1.010
4,0,1895,1,0,80,12,NaN,22.0,0.5,1698,0.701,1.034


In [ ]:
data.tail()

,Group,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
368,0,842,1,0,82,16,1.0,28.0,0.5,1693,0.694,1.037
369,0,2297,1,0,86,16,1.0,26.0,0.5,1688,0.675,1.040
370,1,0,0,0,61,13,2.0,30.0,0.0,1319,0.801,1.331
371,1,763,0,0,63,13,2.0,30.0,0.0,1327,0.796,1.323
372,1,1608,0,0,65,13,2.0,30.0,0.0,1333,0.801,1.317


In [ ]:
data.shape

(373, 12)

In [ ]:
data.isnull().sum()

Group        0
MR Delay     0
M/F          0
Hand         0
Age          0
EDUC         0
SES         19
MMSE         2
CDR          0
eTIV         0
nWBV         0
ASF          0
dtype: int64

In [ ]:
#pengisian data kosong menggunakan nilai modus & mean
modus = data['SES'].mode()[0]
data['SES']=data['SES'].fillna(modus)
data['MMSE']=data['MMSE'].fillna(data['MMSE'].mean())

In [ ]:
data.isna().sum()

Group       0
MR Delay    0
M/F         0
Hand        0
Age         0
EDUC        0
SES         0
MMSE        0
CDR         0
eTIV        0
nWBV        0
ASF         0
dtype: int64

In [ ]:
#duplicate data check
data.duplicated().sum()

0

In [ ]:
data.Group.value_counts()

1    190
0    183
Name: Group, dtype: int64

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 373 entries, 0 to 372
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Group     373 non-null    int64  
 1   MR Delay  373 non-null    int64  
 2   M/F       373 non-null    int64  
 3   Hand      373 non-null    int64  
 4   Age       373 non-null    int64  
 5   EDUC      373 non-null    int64  
 6   SES       373 non-null    float64
 7   MMSE      373 non-null    float64
 8   CDR       373 non-null    float64
 9   eTIV      373 non-null    int64  
 10  nWBV      373 non-null    float64
 11  ASF       373 non-null    float64
dtypes: float64(5), int64(7)
memory usage: 35.1 KB


In [ ]:
#normalisasi data menggunakan min-max norm
data['MMSE']=(data['MMSE']-data['MMSE'].min())/(data['MMSE'].max()-data['MMSE'].min())
data['eTIV']=(data['eTIV']-data['eTIV'].min())/(data['eTIV'].max()-data['eTIV'].min())

In [ ]:
data.shape

(373, 12)

In [ ]:
data.describe(include='all')

,Group,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
count,373.000000,373.000000,373.000000,373.0,373.000000,373.000000,373.000000,373.000000,373.000000,373.000000,373.000000,373.000000
mean,0.509383,595.104558,0.428954,0.0,77.013405,14.597855,2.436997,0.897781,0.290885,0.425533,0.729568,1.195461
std,0.500583,635.485118,0.495592,0.0,7.640957,2.876339,1.109307,0.141282,0.374557,0.196146,0.037135,0.138092
min,0.000000,0.000000,0.000000,0.0,60.000000,6.000000,1.000000,0.000000,0.000000,0.000000,0.644000,0.876000
25%,0.000000,0.000000,0.000000,0.0,71.000000,12.000000,2.000000,0.884615,0.000000,0.279510,0.700000,1.099000
50%,1.000000,552.000000,0.000000,0.0,77.000000,15.000000,2.000000,0.961538,0.000000,0.405345,0.729000,1.194000
75%,1.000000,873.000000,1.000000,0.0,82.000000,16.000000,3.000000,1.000000,0.500000,0.546771,0.756000,1.293000
max,1.000000,2639.000000,1.000000,0.0,98.000000,23.000000,5.000000,1.000000,2.000000,1.000000,0.837000,1.587000


In [ ]:
x=data.iloc[:,data.columns!='Group']
y=data.iloc[:,data.columns=='Group']

In [ ]:
x.shape

(373, 11)

In [ ]:
y.shape

(373, 1)

In [ ]:
x.head()

,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,0,1,0,87,14,2.0,0.884615,0.0,0.981069,0.696,0.883
1,457,1,0,88,14,2.0,1.000000,0.0,1.000000,0.681,0.876
2,0,1,0,75,12,2.0,0.730769,0.5,0.636971,0.736,1.046
3,560,1,0,76,12,2.0,0.923077,0.5,0.703786,0.713,1.010
4,1895,1,0,80,12,2.0,0.692308,0.5,0.659243,0.701,1.034


In [ ]:
y.head()

,Group
0,1
1,1
2,0
3,0
4,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,recall_score, roc_curve, auc

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(x,y,test_size=0.2)

In [ ]:
xtrain.head()

,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
177,0,0,0,82,12,4.0,0.846154,0.5,0.183742,0.695,1.381
101,1233,1,0,69,16,1.0,0.000000,1.0,0.662584,0.676,1.032
17,0,0,0,66,12,3.0,1.000000,0.5,0.379733,0.769,1.213
220,2002,0,0,75,16,1.0,1.000000,0.5,0.348552,0.731,1.236
344,700,1,0,72,16,4.0,0.923077,0.5,0.768374,0.732,0.977


In [ ]:
ytrain.head()

,Group
177,0
101,0
17,0
220,0
344,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier

# PERCENTAGE SPLIT

**C4.5 PERCENTAGE SPLIT VALIDATION**

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y,test_size=0.2, random_state=1)
tree_data = DecisionTreeClassifier(random_state=1)
tree_data.fit(x_train,y_train)

DecisionTreeClassifier(random_state=1)

In [ ]:
y_pred = tree_data.predict(x_test)
cm = confusion_matrix(y_test,y_pred)
print ("Confusion Matrix")
print (cm)
akurasi = classification_report(y_test,y_pred)
print ("Tingkat Akurasi Algoritma C4.5 dengan percentage split")
print ("Akurasi : ", akurasi)
akurasi = accuracy_score(y_test,y_pred)
print ("Tingkat akurasi : %d persen" %(akurasi*100))

Confusion Matrix
[[28  4]
 [ 5 38]]
Tingkat Akurasi Algoritma C4.5 dengan percentage split
Akurasi :                precision    recall  f1-score   support

           0       0.85      0.88      0.86        32
           1       0.90      0.88      0.89        43

    accuracy                           0.88        75
   macro avg       0.88      0.88      0.88        75
weighted avg       0.88      0.88      0.88        75

Tingkat akurasi : 88 persen


In [ ]:
from sklearn.tree import export_graphviz
export_graphviz(tree_data, out_file="Tree_Category.dot", class_names=["0","1","2"],
                feature_names=x.columns, impurity=False, filled=True)

In [ ]:
import graphviz

with open("Tree_Category.dot") as fig:
  dot_graph = fig.read()
  graph = graphviz.Source(dot_graph)

graph.view()

'Source.gv.pdf'

**ADABOOST PERCENTAGE SPLIT**

In [ ]:
# Buat model Adaboost dengan Decision Tree Classifier sebagai estimator
x_train, x_test, y_train, y_test = train_test_split(x, y,test_size=0.2, random_state=1)
adaboost = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2), n_estimators=373)
# Menghitung akurasi dan confusion matrix pada testing set
adaboost.fit(x_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                   n_estimators=373)

In [ ]:
y_pred_test = adaboost.predict(x_test)
acc = accuracy_score(y_test, y_pred_test)
cm_test = confusion_matrix(y_test, y_pred_test)

In [ ]:
print ("Confusion Matrix")
print (cm_test)
akurasi = classification_report(y_test,y_pred_test)
print ("Tingkat Akurasi Algoritma AdaBoost dengan Percentage Split")
print (akurasi)
print ("Tingkat akurasi : %d persen" %(acc*100))

Confusion Matrix
[[27  5]
 [ 3 40]]
Tingkat Akurasi Algoritma AdaBoost dengan Percentage Split
              precision    recall  f1-score   support

           0       0.90      0.84      0.87        32
           1       0.89      0.93      0.91        43

    accuracy                           0.89        75
   macro avg       0.89      0.89      0.89        75
weighted avg       0.89      0.89      0.89        75

Tingkat akurasi : 89 persen


# K-FOLD CROSS VALIDATION

**C4.5 K-FOLD CROSS VALIDATION**

In [ ]:
#melakukan cross validation
x_train, x_test, y_train, y_test = train_test_split(x, y)
classifier = DecisionTreeClassifier(random_state=10)
classifier.fit(x_train,y_train)

DecisionTreeClassifier(random_state=10)

In [ ]:
#menghitung akurasi cross validation
scores= cross_val_score(classifier, x_train, y_train, scoring ='accuracy', cv=10)
scores

array([1.        , 0.92857143, 0.96428571, 0.85714286, 0.89285714,
       0.92857143, 0.71428571, 1.        , 0.92857143, 0.92592593])

In [ ]:
pred = cross_val_predict(classifier, x, y, cv=10)
pred

array([1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1,
       1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1,

In [ ]:
#Menghitung akurasi
from sklearn.metrics import confusion_matrix
confusion_matrix = confusion_matrix(y, pred)
print(confusion_matrix)

[[168  15]
 [ 25 165]]


In [ ]:
print(classification_report(y, pred))

              precision    recall  f1-score   support

           0       0.87      0.92      0.89       183
           1       0.92      0.87      0.89       190

    accuracy                           0.89       373
   macro avg       0.89      0.89      0.89       373
weighted avg       0.89      0.89      0.89       373



In [ ]:
#K-Fold Cross Validation
print(f'Tingkat akurasi: {(accuracy_score(y, pred)) * 100}%')


Tingkat akurasi: 89.27613941018767%


In [ ]:
from sklearn.tree import export_graphviz
export_graphviz(tree_data, out_file="Tree_Category.dot", class_names=["0","1","2"],
                feature_names=x.columns, impurity=False, filled=True)

In [ ]:
import graphviz

with open("Tree_Category.dot") as fig:
  dot_graph = fig.read()
  graph = graphviz.Source(dot_graph)

graph.view()

'Source.gv.pdf'

**ADABOOST KFOLD CROSS VALIDATION**

In [ ]:
#melakukan cross validation
x_train, x_test, y_train, y_test = train_test_split(x, y)
adaboostfold = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2), n_estimators=373)
adaboostfold.fit(x_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                   n_estimators=373)

In [ ]:
#menghitung akurasi cross validation
score= cross_val_score(adaboostfold, x, y, scoring ='accuracy', cv=10)
score

array([0.92105263, 0.86842105, 0.86842105, 0.91891892, 0.91891892,
       0.94594595, 0.89189189, 0.89189189, 0.97297297, 0.91891892])

In [ ]:
y_predict = cross_val_predict(adaboostfold, x, y, cv=10)
y_predict

array([1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1,
       1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0,

In [ ]:
#Menghitung akurasi
from sklearn.metrics import confusion_matrix
confusion_matrix = confusion_matrix(y, y_predict)
print(confusion_matrix)

[[165  18]
 [ 15 175]]


In [ ]:
print(classification_report(y, y_predict))
accc = accuracy_score(y, y_predict)
print(f'Tingkat akurasi: {accc:.2f}%')

              precision    recall  f1-score   support

           0       0.92      0.90      0.91       183
           1       0.91      0.92      0.91       190

    accuracy                           0.91       373
   macro avg       0.91      0.91      0.91       373
weighted avg       0.91      0.91      0.91       373

Tingkat akurasi: 0.91%


In [ ]:
print(f'Tingkat akurasi: {(accuracy_score(y, y_predict)) * 100}%')

Tingkat akurasi: 91.15281501340483%
